In [1]:
"""
Model Evaluation Script - v1.0 (KNN Model)

This script evaluates the current production model (v1.0) to assess:
1. Model performance metrics
2. Generalization capability
3. Overfitting/underfitting
4. Prediction quality
5. Areas for improvement

Usage:
    python scripts/evaluate_model.py
"""

import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import model_selection, metrics
from pathlib import Path


In [4]:

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('='*70)
print('MODEL EVALUATION - v1.0 (KNN)')
print('='*70)
print()

# ============================================================================
# 1. LOAD MODEL AND DATA
# ============================================================================
print('1. Loading model and data...')

MODEL_PATH = Path('../models/v1.0/model.pkl')
FEATURES_PATH = Path('../models/v1.0/model_features.json')

with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

with open(FEATURES_PATH, 'r') as f:
    required_features = json.load(f)

print(f'   ✓ Model loaded: {type(model).__name__}')
print(f'   ✓ Number of features: {len(required_features)}')
print(f'   ✓ Pipeline steps: {" → ".join([step[0] for step in model.steps])}')

# Load training data
house_data = pd.read_csv('../data/kc_house_data.csv')
demographics = pd.read_csv('../data/zipcode_demographics.csv', dtype={'zipcode': str})

print(f'   ✓ House data: {house_data.shape[0]:,} samples')
print(f'   ✓ Demographics: {demographics.shape[0]} zipcodes')

# Prepare data
house_data['zipcode'] = house_data['zipcode'].astype(str)
merged_data = house_data.merge(demographics, on='zipcode', how='left')
merged_data = merged_data.drop(columns=['id', 'date', 'zipcode'])

X = merged_data[required_features]
y = merged_data['price']

print(f'   ✓ Features shape: {X.shape}')
print(f'   ✓ Missing values: {X.isnull().sum().sum()}')
print()

# ============================================================================
# 2. TRAIN-TEST SPLIT
# ============================================================================
print('2. Creating train-test split...')

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f'   ✓ Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/X.shape[0]:.1%})')
print(f'   ✓ Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/X.shape[0]:.1%})')
print()

# ============================================================================
# 3. MODEL PERFORMANCE METRICS
# ============================================================================
print('3. Calculating performance metrics...')

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_r2 = metrics.r2_score(y_train, y_train_pred)
test_r2 = metrics.r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(metrics.mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(metrics.mean_squared_error(y_test, y_test_pred))

train_mae = metrics.mean_absolute_error(y_train, y_train_pred)
test_mae = metrics.mean_absolute_error(y_test, y_test_pred)

print()
print('   ' + '='*66)
print('   MODEL PERFORMANCE METRICS')
print('   ' + '='*66)
print(f'   R² Score:')
print(f'     Training:   {train_r2:.4f}')
print(f'     Test:       {test_r2:.4f}')
print(f'     Difference: {train_r2 - test_r2:.4f} ({(train_r2 - test_r2)/train_r2*100:.1f}% drop)')

print(f'   \n   RMSE (Root Mean Squared Error):')
print(f'     Training:   ${train_rmse:,.0f}')
print(f'     Test:       ${test_rmse:,.0f}')
print(f'     Difference: ${test_rmse - train_rmse:,.0f}')

print(f'   \n   MAE (Mean Absolute Error):')
print(f'     Training:   ${train_mae:,.0f}')
print(f'     Test:       ${test_mae:,.0f}')
print(f'     Difference: ${test_mae - train_mae:,.0f}')

print(f'   \n   Mean house price: ${y.mean():,.0f}')
print(f'   Test RMSE as % of mean: {test_rmse/y.mean()*100:.1f}%')
print('   ' + '='*66)
print()

# ============================================================================
# 4. OVERFITTING ANALYSIS
# ============================================================================
print('4. Analyzing overfitting...')

r2_gap = train_r2 - test_r2
r2_gap_pct = (r2_gap / train_r2) * 100

print(f'   R² Gap: {r2_gap:.4f} ({r2_gap_pct:.1f}%)')

if r2_gap < 0.05:
    print('   ✓ Model shows GOOD generalization (gap < 5%)')
    print('     The model performs similarly on training and test data.')
elif r2_gap < 0.15:
    print('   ⚠ Model shows MODERATE overfitting (5% < gap < 15%)')
    print('     The model performs noticeably better on training data.')
    print('     Consider: regularization, simpler model, more data.')
else:
    print('   ✗ Model shows SIGNIFICANT overfitting (gap > 15%)')
    print('     The model has memorized training data.')
    print('     Action needed: regularization, feature selection, simpler model.')
print()


MODEL EVALUATION - v1.0 (KNN)

1. Loading model and data...
   ✓ Model loaded: Pipeline
   ✓ Number of features: 33
   ✓ Pipeline steps: robustscaler → kneighborsregressor
   ✓ House data: 21,613 samples
   ✓ Demographics: 70 zipcodes
   ✓ Features shape: (21613, 33)
   ✓ Missing values: 0

2. Creating train-test split...
   ✓ Training set: 16,209 samples (75.0%)
   ✓ Test set: 5,404 samples (25.0%)

3. Calculating performance metrics...

   MODEL PERFORMANCE METRICS
   R² Score:
     Training:   0.8414
     Test:       0.7281
     Difference: 0.1133 (13.5% drop)
   
   RMSE (Root Mean Squared Error):
     Training:   $143,467
     Test:       $201,661
     Difference: $58,194
   
   MAE (Mean Absolute Error):
     Training:   $76,234
     Test:       $102,055
     Difference: $25,821
   
   Mean house price: $540,088
   Test RMSE as % of mean: 37.3%

4. Analyzing overfitting...
   R² Gap: 0.1133 (13.5%)
   ⚠ Model shows MODERATE overfitting (5% < gap < 15%)
     The model performs not